# Gemini Function Calling — Two Functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/gemini_two_functions.ipynb)

> **How to activate the badge:**
> 1. Upload this `.ipynb` file to your GitHub repo
> 2. Replace `YOUR_GITHUB_USERNAME` and `YOUR_REPO_NAME` in the link above
> 3. The badge will then open this notebook directly in Colab ✅

> **Or just drag & drop** this file at [colab.research.google.com](https://colab.research.google.com) → *File → Upload notebook*

In [ ]:
!pip install google-generativeai -q
import os
import google.generativeai as genai

# Setup
GOOGLE_API_KEY = "your-key"   # Replace with your key
genai.configure(api_key=GOOGLE_API_KEY)

## Step 1 — Define Two Python Functions

In [ ]:
# Function 1: Math
def calculate(operation, num1, num2):
    if operation == "add":      return num1 + num2
    if operation == "subtract": return num1 - num2
    if operation == "multiply": return num1 * num2
    if operation == "divide":   return num1 / num2 if num2 != 0 else "Error: divide by zero"

# Function 2: Unit Conversion
def convert_units(value, from_unit, to_unit):
    conversions = {
        ("km",         "miles"):       value * 0.621371,
        ("miles",      "km"):          value * 1.60934,
        ("kg",         "lbs"):         value * 2.20462,
        ("lbs",        "kg"):          value / 2.20462,
        ("celsius",    "fahrenheit"):  (value * 9/5) + 32,
        ("fahrenheit", "celsius"):     (value - 32) * 5/9,
    }
    result = conversions.get((from_unit.lower(), to_unit.lower()))
    return f"{value} {from_unit} = {round(result, 2)} {to_unit}" if result else "Unsupported conversion"

# Quick test
print(calculate("add", 10, 5))               # 15
print(convert_units(100, "km", "miles"))     # 62.14 miles

## Step 2 — Tell Gemini About the Functions (Tool Schema)

In [ ]:
calculate_tool = genai.types.FunctionDeclaration(
    name="calculate",
    description="Do math: add, subtract, multiply, or divide two numbers.",
    parameters={
        "type": "object",
        "properties": {
            "operation": {"type": "string", "description": "add / subtract / multiply / divide"},
            "num1":      {"type": "number", "description": "First number"},
            "num2":      {"type": "number", "description": "Second number"}
        },
        "required": ["operation", "num1", "num2"]
    }
)

convert_tool = genai.types.FunctionDeclaration(
    name="convert_units",
    description="Convert units: km/miles, kg/lbs, celsius/fahrenheit.",
    parameters={
        "type": "object",
        "properties": {
            "value":     {"type": "number", "description": "Value to convert"},
            "from_unit": {"type": "string", "description": "e.g. km, kg, celsius"},
            "to_unit":   {"type": "string", "description": "e.g. miles, lbs, fahrenheit"}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
)

## Step 3 — Create Model & Chat

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash", tools=[calculate_tool, convert_tool])
chat  = model.start_chat()

# Map function name to actual Python function
FUNCTIONS = {
    "calculate":     calculate,
    "convert_units": convert_units
}

## Step 4 — Chat Loop

**Try these prompts:**
- `What is 25 + 17?` → uses **calculate only**
- `Convert 100 km to miles` → uses **convert_units only**
- `Add 50+30 AND convert 37 celsius to fahrenheit` → uses **both functions**

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    response = chat.send_message(user_input)

    # Loop through all response parts and check for function calls
    function_was_called = False
    for part in response.candidates[0].content.parts:
        fc = part.function_call
        if fc and fc.name:
            function_was_called = True
            args   = dict(fc.args)
            result = FUNCTIONS[fc.name](**args)   # call the matching function
            print(f"[{fc.name}] {args} → {result}")

    # If no function was called, print Gemini's plain text reply
    if not function_was_called:
        print(f"Gemini: {response.text}")